<a href="https://colab.research.google.com/github/Longhanhmid24/DoAn_Email/blob/main/TinyLLama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
base_path = "/content/drive/MyDrive/Deep_Learning_Email/dataset_split/"
train_df = pd.read_csv(base_path + "train.csv")
val_df   = pd.read_csv(base_path + "val.csv")
test_df  = pd.read_csv(base_path + "test.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate

In [5]:
import sys
!{sys.executable} -m pip install --upgrade pip
# Perform a clean uninstall of potentially conflicting packages
!{sys.executable} -m pip uninstall -y numpy pyarrow datasets pandas

# Reinstall specific versions to ensure compatibility and prevent binary incompatibility errors
!{sys.executable} -m pip install numpy==1.26.4 pyarrow==14.0.2 datasets==2.19.0 pandas==2.2.2

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pyarrow 23.0.1
Uninstalling pyarrow-23.0.1:
  Successfully uninstalled pyarrow-23.0.1
Found existing installation: datasets 4.8.4
Uninstalling datasets-4.8.4:
  Successfully uninstalled datasets-4.8.4
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pyarrow-14.0.2-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached datasets-2.19.0-py3-none-any.whl.metadata (19 kB)
  Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached pyarrow-14.0.2-cp312-cp312-manylinux_2_28_x86_64.whl (38.0 MB)
Using cached datasets-

In [6]:
from datasets import Dataset
from transformers import AutoTokenizer

# 1. Chuyển Pandas DataFrame sang Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# 2. Khởi tạo Tokenizer của TinyLlama
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Đặt pad_token bằng eos_token để tránh lỗi padding
tokenizer.pad_token = tokenizer.eos_token

# 3. Hàm định dạng dữ liệu (Prompt Formatting)
def format_prompt(example):
    # Sử dụng format Chat của TinyLlama để tận dụng tối đa sức mạnh của phiên bản Chat
    text = f"<|system|>\nYou are a helpful email assistant.</s>\n<|user|>\n{example['Input_Text']}</s>\n<|assistant|>\n{example['Output_Text']}</s>"
    return {"text": text}

# 4. Áp dụng định dạng cho toàn bộ tập dữ liệu
train_dataset = train_dataset.map(format_prompt)
val_dataset = val_dataset.map(format_prompt)

# Xem thử một mẫu dữ liệu sau khi format
print(train_dataset[0]['text'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Map:   0%|          | 0/5116 [00:00<?, ? examples/s]

Map:   0%|          | 0/639 [00:00<?, ? examples/s]

<|system|>
You are a helpful email assistant.</s>
<|user|>
context:  | email: yes i am here for the time being i am awaiting word on the sale of the trade group i have been interviewing a little and just hanging out how are you have you been interviewing i am leaving right now to go interview with rwe a german utility starting up a north america energy trading group send me your phone numbers so that i can call you sometime my home number is i hope everything is well and hope to talk to you soon john</s>
<|assistant|>
johnny what do you mean you are there for the time being would not your position be maintained when the trading group is acquired i would think that as long as j arnold has a say you would be included am i incorrect things in unemployment land are pretty surreal when i stop and think about the hours that i put in only to be laid offgo figure however i try not to dwell in areas where i had no control i have been flooding my resume throughout the houstonbased energy compani

In [19]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# 1. Cấu hình Quantization (4-bit) để chạy mượt trên Colab
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # Changed to float16 for broader compatibility
)

# 2. Tải mô hình TinyLlama Base
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)
model.config.use_cache = False # Tắt cache để tiết kiệm RAM khi training

# 3. Cấu hình LoRA (Chỉ huấn luyện các tham số quan trọng)
lora_config = LoraConfig(
    r=64, # Rank càng cao học càng nhiều nhưng tốn RAM hơn
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [24]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer



# 4. Thiết lập các tham số huấn luyện (Training Arguments)
training_args = TrainingArguments(
    output_dir="./tinyllama-email-finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    max_grad_norm=1.0,
    optim="paged_adamw_8bit",
    report_to="none"
)
# 5. Khởi tạo SFTTrainer (Supervised Fine-Tuning)
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args,
)

# 6. Bắt đầu quá trình huấn luyện
trainer.train()

# 7. Lưu lại bộ trọng số LoRA sau khi huấn luyện xong
trainer.model.save_pretrained("/content/drive/MyDrive/Deep_Learning_Email/tinyllama_lora_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Deep_Learning_Email/tinyllama_lora_model")
print("Đã lưu model thành công vào Google Drive!")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/5116 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5116 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/639 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/639 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,2.774772,2.715996
2,2.650600,2.673034
3,2.584487,2.698779
4,2.530010,2.676142
5,2.475632,2.674578
6,2.426250,2.672830
7,2.383179,2.674411
8,2.352715,2.686610
9,2.334587,2.691988
10,2.324591,2.694217


Đã lưu model thành công vào Google Drive!


#### Đánh giá mô hình

In [25]:
import time
import torch
from tqdm import tqdm

# Đảm bảo mô hình đang ở chế độ đánh giá
model.eval()

predictions = []
references = test_df['Output_Text'].astype(str).tolist()
inputs = test_df['Input_Text'].astype(str).tolist()

start_time = time.time()

# Duyệt qua từng mẫu trong tập test
for input_text in tqdm(inputs, desc="Đang tạo văn bản..."):
    # Format prompt chuẩn như lúc train
    prompt = f"<|system|>\nYou are a helpful email assistant.</s>\n<|user|>\n{input_text}</s>\n<|assistant|>\n"
    inputs_tensor = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs_tensor,
            max_new_tokens=100, # Giới hạn số token sinh ra để tránh sinh dài dòng
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.7, # Tuỳ chỉnh độ sáng tạo
            do_sample=True
        )

    # Giải mã kết quả
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Chỉ lấy phần câu trả lời của trợ lý (cắt bỏ phần prompt ban đầu)
    if "<|assistant|>" in generated_text:
        answer = generated_text.split("<|assistant|>")[-1].strip()
    else:
        answer = generated_text.strip()

    predictions.append(answer)

end_time = time.time()

# Tính thời gian inference trung bình cho 1 mẫu
total_inference_time = end_time - start_time
avg_inference_time = total_inference_time / len(inputs)
print(f"Hoàn thành! Thời gian inference trung bình: {avg_inference_time:.4f} giây/mẫu")

Đang tạo văn bản...:   0%|          | 2/640 [00:13<1:13:27,  6.91s/it]


KeyboardInterrupt: 